# بناء تطبيقات توليد الصور (OpenAI)

هناك أكثر في نماذج اللغات الكبيرة من مجرد توليد النصوص. يمكنك أيضًا توليد صور من أوصاف نصية. الصور كوسيط مفيدة في مجالات مثل تقنية الطب، الهندسة المعمارية، السياحة، تطوير الألعاب، التسويق، والمزيد. في هذا الدرس نعمل مع نماذج **GPT Image** الحديثة على منصة OpenAI.

## أهداف التعلم

بنهاية هذا الدرس ستتمكن من:

- شرح ما هو توليد الصور وأين يكون مفيدًا.
- فهم عائلة النماذج `gpt-image` وكيف تختلف عن نماذج DALL·E القديمة.
- بناء تطبيق لتوليد الصور وتحرير الصور.

## ما هو توليد الصور؟

تقوم نماذج توليد الصور بإنشاء صور بناءً على مدخل نصي. تتعلم النماذج الحديثة مثل `gpt-image` العلاقة بين النص والصور أثناء التدريب، ثم تحول الضوضاء العشوائية تدريجيًا إلى صورة تطابق مدخل النص الخاص بك.

هناك عائلتان شهيرتان من نماذج الصور:

- **`gpt-image` (OpenAI)** — الجيل الحالي المستخدم في هذا الدرس. يدعم توليد الصور من النص وتحرير الصور (إعادة الرسم مع قناع).
- **Midjourney** — نموذج طرف ثالث شائع له خدمته الخاصة وسير عمل قائم على Discord.

> نماذج الصور القديمة من OpenAI — **DALL·E 2** و **DALL·E 3** — تعتبر قديمة. DALL·E 3 لم يعد متاحًا للنشر الجديد، والميزات مثل `create_variation` كانت موجودة فقط في DALL·E 2. استخدم نماذج `gpt-image` للتطبيقات الجديدة.

> **مهم:** نماذج `gpt-image` تُرجع الصورة المُولَّدة كـ **base64** (`b64_json`)، وليس كرابط URL. يقوم كودك بفك ترميز سلسلة base64 إلى بايتات ويحفظها — لا يوجد رابط صورة للتحميل.


## بناء تطبيقك الأول لتوليد الصور

فما المطلوب لبناء تطبيق لتوليد الصور؟ تحتاج إلى المكتبات التالية:

- **python-dotenv**، يُنصح بشدة باستخدام هذه المكتبة للاحتفاظ بأسرارك في ملف *.env* بعيدًا عن الكود.
- **openai**، هذه المكتبة هي التي ستستخدمها للتفاعل مع واجهة برمجة تطبيقات OpenAI.
- **pillow**، للعمل مع الصور في بايثون.


1. أنشئ ملف *.env* بالمحتوى التالي:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. اجمع المكتبات أعلاه في ملف يسمى *requirements.txt* كما يلي:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. بعد ذلك، أنشئ بيئة افتراضية وقم بتثبيت المكتبات:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!ملاحظة]
> بالنسبة لنظام ويندوز، استخدم الأوامر التالية لإنشاء وتفعيل بيئة افتراضية:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. أضف الكود التالي في ملف يسمى *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Create OpenAI object (reads OPENAI_API_KEY from your .env)
    client = openai.OpenAI()


    try:
        # Create an image by using the image generation API
        generation_response = client.images.generate(
            model="gpt-image-2",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
            size='1024x1024',
            n=1
        )
        # Set the directory for the stored image
        image_dir = os.path.join(os.curdir, 'images')

        # If the directory doesn't exist, create it
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialize the image path (note the filetype should be png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image models return the image as base64 (b64_json), not a URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Display the image in the default image viewer
        image = Image.open(image_path)
        image.show()

    # catch exceptions
    except openai.BadRequestError as err:
        print(err)

    ```

دعنا نشرح هذا الكود:

- أولاً، نستورد المكتبات التي نحتاجها، بما في ذلك مكتبة OpenAI، ومكتبة dotenv، ووحدة base64، ومكتبة Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- بعد ذلك، ننشئ العميل، الذي يقرأ مفتاح API من ``.env`` الخاص بك.

    ```python
    # Create OpenAI object
    client = openai.OpenAI()
    ```

- بعد ذلك، نولد الصورة:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-2",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` النماذج تُرجع الصورة كسلسلة **base64** داخل `data[0].b64_json`. نقوم بفك ترميزها إلى بايتات ونكتبها في ملف — لا يوجد رابط URL للتحميل.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- أخيراً، نفتح الصورة ونستخدم عارض الصور الافتراضي لعرضها:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### المزيد من التفاصيل حول توليد الصورة

لننظر إلى معايير `images.generate`:

- **model**، هو نموذج الصورة المستخدم (على سبيل المثال `gpt-image-2`).
- **prompt**، هو الموجه النصي المستخدم لتوليد الصورة. هنا هو "أرنب على حصان، يحمل مصاصة، على مرج ضبابي ينمو فيه النرجس".
- **size**، هو حجم الصورة المولدة (على سبيل المثال `1024x1024`، `1536x1024`، `1024x1536`، أو `"auto"`).
- **n**، هو عدد الصور التي يتم توليدها. هنا نقوم بتوليد صورة واحدة.

> نماذج الصور لا تأخذ معلمة `temperature` — هذه خاصية تحكم في توليد النصوص. للحصول على تنوع، استدعِ الـ API مرة أخرى؛ لتقليل التنوع، اجعل الموجه أكثر تحديداً.

## قدرات إضافية لتوليد الصور

لقد رأيت كيف تولد صورة ببضع أسطر من بايثون. نماذج `gpt-image` يمكنها أيضاً **تعديل** صورة موجودة. من خلال توفير صورة، وقناع اختياري **mask** (يحدد المنطقة المراد تغييرها)، وموجه، يمكنك تغيير جزء من صورة — على سبيل المثال، إضافة قبعة لأرانبنا.

```python
response = client.images.edit(
  model="gpt-image-2",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# edits are also returned as base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

الصورة الأساسية تحتوي فقط على الأرنب؛ الصورة النهائية بها القبعة على الأرنب.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
